# TP-3 — TinyGPT: export a ONNX y un inference server

Autor: Msc. Abraham R.

Resolución: Fermín Rodríguez del Castillo — Maestría en Inteligencia Artificial (FIUBA)

Vas a tomar el TinyGPT que fine-tuneaste en el TP-2 y ponerlo en producción: sacarlo de
PyTorch, meterlo en un grafo de cómputo portable, y ponerlo detrás de una API compatible
con OpenAI.

## De un framework de entrenamiento a un runtime

Acá no se entrena nada. El TP-1 preentrenó un modelo y el TP-2 le enseñó a seguir
instrucciones; los dos corrieron dentro de un notebook, en PyTorch, un request por vez.
Producción no funciona así, porque un framework de entrenamiento está hecho para ser
flexible y un runtime está hecho para una sola tarea:

| | PyTorch | un runtime de inferencia |
|---|---|---|
| Hecho para | experimentar, autograd | un grafo fijo, ejecutado rápido |
| El modelo es | código Python + pesos | un grafo serializado, sin Python |
| Se entrega como | un repo y un entorno | un solo archivo |
| Corre en | donde instale PyTorch | CPU, GPU, NPU, teléfono, browser |

Este es el paso que hay detrás de runtimes como:
- ONNX Runtime (el que vas a usar acá)
- TensorRT, OpenVINO, CoreML
- llama.cpp / GGUF, que recorrimos en clase

**Parte 1 — Export.** Convertí el modelo en un grafo ONNX y demostrá que el grafo calcula
lo que calculaba PyTorch. Después exportá un segundo grafo que lleve el KV-cache.

**Parte 2 — Cuantizar y medir.** Hacelo 4x más chico, y después averiguá honestamente qué
costó eso en calidad y qué compró en velocidad.

**Parte 3 — Servir.** Poné el grafo detrás de `POST /v1/chat/completions` y llamalo con el
cliente oficial de OpenAI.

El modelo no cambia en este notebook. Es el mismo checkpoint de punta a punta — lo único
que cambia es su representación.

## Consignas

**Parte 1**
- Consigna I — exportar a ONNX, y escribir el test de paridad que lo demuestra.
- Consigna II — exportar un segundo grafo que reciba y devuelva el **KV-cache**.
- Consigna III — portar tu `generateV2` del TP-1 al runtime: temperatura, top-k, top-p, y
  el cache.

**Parte 2**
- Consigna IV — cuantizar a **INT8**, y medir qué costó en comportamiento.
- Consigna V — benchmarkear PyTorch vs ONNX vs INT8, y separar el tiempo de engine del de
  Python.

**Parte 3**
- Consigna VI — implementar `POST /v1/chat/completions`, compatible con OpenAI.
- Consigna VII — agregar **streaming**: el mismo endpoint, token por token sobre SSE.
- Consigna VIII — llamar a tu server con el cliente oficial `openai`, de las dos formas.

## Cómo trabajar en esto

**Requisitos previos:** el TP-2. Este notebook carga
`./checkpoints/tp2_full_ft/checkpoint_final.pt`. Sin ese archivo todo sigue corriendo,
con pesos al azar — los números siguen teniendo sentido, el texto generado no.

Las Consignas I, II, III, IV y VI vienen con una celda `check_*()`. Corréla antes de
seguir. Los bugs de paridad son silenciosos: un grafo equivocado devuelve números
plausibles, y no te vas a enterar hasta que las generaciones estén sutilmente mal.

**Acá las versiones importan.** `torch.onnx.export` es una API en movimiento, y el
comportamiento del exportador cambia entre releases. Las versiones contra las que se
escribió este notebook las imprime la primera celda; si las tuyas son distintas y algo se
rompe, eso es lo primero que hay que mirar.

Referencias: [ONNX](https://onnx.ai/) ·
[ONNX Runtime](https://onnxruntime.ai/docs/) ·
[torch.onnx](https://pytorch.org/docs/stable/onnx.html) ·
[chat completions de OpenAI](https://platform.openai.com/docs/api-reference/chat)

In [1]:
import json
import os
import re
import threading
import time
import uuid
from collections import Counter
from typing import Dict, List, Literal, Optional, Sequence, Tuple

import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer

import onnx
import onnxruntime as ort
import requests
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

from tinygpt import (
    GPTConfig,
    TinyGPT,
    describe,
    load_checkpoint,
    load_tinyshakespeare,
)

torch.manual_seed(1337)

# Todo acá corre en CPU a propósito: es el target que este notebook puede medir
# honestamente, y el que todo el mundo tiene.
print(f"torch        {torch.__version__}")
print(f"onnx         {onnx.__version__}")
print(f"onnxruntime  {ort.__version__}")
print(f"providers    {ort.get_available_providers()}")

/Users/fermin/Desktop/UBA/PLNII/tps-plnii/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.2


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


torch        2.2.2
onnx         1.23.0
onnxruntime  1.23.2
providers    ['CoreMLExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']


# Etapa 0 — el modelo que vas a poner en producción

El fine-tune del TP-2, cargado exactamente como el TP-2 lo dejó: mismo tokenizer, mismos
cuatro tokens especiales, mismo embedding y head sin atar.

In [2]:
USER, ASSISTANT, END, PAD = "<|user|>", "<|assistant|>", "<|end|>", "<|pad|>"
TP2_CKPT = "./checkpoints/tp2_full_ft/checkpoint_final.pt"
ONNX_DIR = "./onnx"
os.makedirs(ONNX_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"additional_special_tokens": [USER, ASSISTANT, END, PAD]})
END_ID = tokenizer.convert_tokens_to_ids(END)
# El andamiaje que una respuesta nunca debe contener -- Consigna VI.
SCAFFOLD_IDS = [tokenizer.convert_tokens_to_ids(t) for t in (USER, ASSISTANT, PAD)]

# tie_weights=False por la misma razón que en el TP-2: los checkpoints mantienen
# token_emb y head como dos matrices separadas.
config = GPTConfig(vocab_size=len(tokenizer), tie_weights=False)
model = TinyGPT(config)

if os.path.exists(TP2_CKPT):
    load_checkpoint(model, TP2_CKPT, map_location="cpu")
    print(f"pesos fine-tuneados cargados desde {TP2_CKPT}")
else:
    print("no se encontró el checkpoint del TP-2 -- se sigue con pesos al azar.")
    print("(todas las mediciones del notebook siguen funcionando; el texto va a ser ruido)")

model.eval()
describe(model, "TinyGPT (fine-tuneado en el TP-2)")
print(f"\nblock_size {config.block_size} | n_layer {config.n_layer} | "
      f"n_head {config.n_head} | vocab {config.vocab_size:,}")

pesos fine-tuneados cargados desde ./checkpoints/tp2_full_ft/checkpoint_final.pt
TinyGPT (fine-tuneado en el TP-2): 6,535,552 parameters | 6,535,552 trainable (100.00%)

block_size 32 | n_layer 2 | n_head 4 | vocab 50,261


## Contra qué medimos el grafo

Se necesitan tres cosas en todo el notebook, y las tres están dadas: texto reservado para
puntuar, un conjunto de prompts, y un loop de decodificación.

**La accuracy no es una de ellas.** El TP-2 midió `who_said` prácticamente en el nivel de
azar sobre 98 personajes, así que una columna de accuracy acá daría 0% para cada artefacto
y no te diría nada sobre lo que hizo la cuantización. En su lugar usamos lo que sí se
mueve:

- **cross-entropy** sobre Shakespeare reservado — el objetivo de preentrenamiento, el
  mismo número que el TP-2 usó para medir el olvido. Es una medida real de calidad y
  funciona por malo que sea el modelo en cualquier tarea puntual;
- **coincidencia** con el modelo fp32 — con qué frecuencia dos artefactos toman la misma
  decisión.

Calidad y fidelidad son preguntas distintas, y después de cuantizar necesitás las dos.

In [3]:
# La misma extracción que el TP-2, sobre el mismo corpus, con el mismo shuffle 1337 --
# así que estos son ejemplos sobre los que el fine-tune no entrenó.
_text = load_tinyshakespeare(None)
_speeches = re.findall(r"^([A-Z][A-Za-z' ]{1,30}):\n((?:.+\n)+)", _text, flags=re.M)
_counts = Counter(s for s, _ in _speeches)
_speeches = [(s, t) for s, t in _speeches if _counts[s] >= 20]
import random as _random
_random.Random(1337).shuffle(_speeches)
_held_out = _speeches[int(0.9 * len(_speeches)):]

BUDGET = config.block_size + 1


def _who_said(speaker: str, speech: str) -> Optional[Tuple[str, str]]:
    line = " ".join(speech.split())
    room = BUDGET - len(tokenizer.encode(f"{USER}who said: {ASSISTANT}")) \
                  - len(tokenizer.encode(speaker + END))
    if room < 5:
        return None
    body = tokenizer.decode(tokenizer.encode(line)[:room])
    return f"{USER}who said: {body}{ASSISTANT}", speaker


EVAL_SET = [p for p in (_who_said(s, t) for s, t in _held_out) if p is not None]

# Texto reservado para la cross-entropy: el último 10% del corpus, en ventanas de block_size.
_ids = tokenizer(_text[-120_000:], add_special_tokens=False)["input_ids"]
LM_WINDOWS = [(np.array([_ids[i:i + config.block_size]], dtype=np.int64),
               np.array(_ids[i + 1:i + config.block_size + 1], dtype=np.int64))
              for i in range(0, 64 * config.block_size, config.block_size)]

# Algunos prompts para generar, para leer más que para puntuar.
DEMO_PROMPTS = [f"{USER}continue: To be, or not to be,{ASSISTANT}",
                f"{USER}who said: Now is the winter of our discontent{ASSISTANT}",
                f"{USER}reverse: sword{ASSISTANT}"]


def make_session(path: str, providers=("CPUExecutionProvider",)) -> ort.InferenceSession:
    """Una sesión de ONNX Runtime con la charlatanería de carga bajada."""
    so = ort.SessionOptions()
    so.log_severity_level = 3
    return ort.InferenceSession(path, so, providers=list(providers))


# Cada llamada a una sesión pasa por acá, así la Consigna V puede preguntar cuánto del
# reloj fue el engine y cuánto fue Python. Usalo en vez de sess.run.
ENGINE = {"seconds": 0.0}


def run_session(sess: ort.InferenceSession, feed: Dict[str, np.ndarray]) -> List[np.ndarray]:
    t0 = time.perf_counter()
    out = sess.run(None, feed)
    ENGINE["seconds"] += time.perf_counter() - t0
    return out


def _shape(value) -> str:
    t = value.type.tensor_type
    dims = [(d.dim_param or str(d.dim_value)) for d in t.shape.dim]
    return f"{onnx.TensorProto.DataType.Name(t.elem_type).lower()}[{','.join(dims)}]"


def show_graph(path: str, max_nodes: int = 12) -> None:
    """Firma, mezcla de operadores y los primeros nodos de un archivo ONNX."""
    g = onnx.load(path).graph
    stored = {i.name for i in g.initializer}
    size = sum(len(i.raw_data) for i in g.initializer) / 1e6
    print(f"{path}  |  {len(g.node)} nodos  |  {len(g.initializer)} tensores de pesos "
          f"({size:.1f} MB)")
    print("  in   " + ", ".join(f"{i.name}: {_shape(i)}" for i in g.input))
    print("  out  " + ", ".join(f"{o.name}: {_shape(o)}" for o in g.output))
    print(f"  ops  {dict(Counter(n.op_type for n in g.node).most_common(8))}")
    for n in g.node[:max_nodes]:
        args = ", ".join(("*" + a if a in stored else a) for a in n.input)
        print(f"    {n.output[0][:28]:<30} = {n.op_type}({args[:52]})")
    if len(g.node) > max_nodes:
        print(f"    ... {len(g.node) - max_nodes} más    (* = un peso guardado en el archivo)")


def greedy_torch(m: nn.Module, prompt: str, max_new_tokens: int = 14) -> str:
    """Loop de decodificación de referencia, sin cache, para comparar peras con peras."""
    ids = tokenizer.encode(prompt)
    produced = []
    for _ in range(max_new_tokens):
        x = torch.tensor(ids[-config.block_size:], dtype=torch.long)[None, :]
        with torch.no_grad():
            nxt = int(m(x)[0, -1].argmax())
        if nxt == END_ID:
            break
        produced.append(nxt)
        ids.append(nxt)
    return tokenizer.decode(produced)


def greedy_onnx(sess: ort.InferenceSession, prompt: str, max_new_tokens: int = 14) -> str:
    """El mismo loop contra una sesión cuya única entrada es `input_ids`."""
    ids = tokenizer.encode(prompt)
    produced = []
    for _ in range(max_new_tokens):
        x = np.array([ids[-config.block_size:]], dtype=np.int64)
        nxt = int(run_session(sess, {"input_ids": x})[0][0, -1].argmax())
        if nxt == END_ID:
            break
        produced.append(nxt)
        ids.append(nxt)
    return tokenizer.decode(produced)


def cross_entropy(next_logits) -> float:
    """
    Cross-entropy media sobre LM_WINDOWS, en nats por token.

    `next_logits(x) -> (T, vocab)` para una ventana `x` de forma (1, T).
    """
    total = 0.0
    for x, y in LM_WINDOWS:
        logits = next_logits(x).astype(np.float64)
        shift = logits - logits.max(axis=-1, keepdims=True)
        logZ = np.log(np.exp(shift).sum(axis=-1))
        total += float((logZ - shift[np.arange(len(y)), y]).mean())
    return total / len(LM_WINDOWS)


def torch_logits(x: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return model(torch.from_numpy(x))[0].numpy()


def onnx_logits(sess: ort.InferenceSession):
    return lambda x: sess.run(None, {"input_ids": x})[0][0]


print(f"{len(EVAL_SET):,} prompts who_said reservados | "
      f"{len(LM_WINDOWS)} ventanas de texto reservado")
print(f"cross-entropy de torch: {cross_entropy(torch_logits):.4f} nats/token")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (38796 > 1024). Running this sequence through the model will result in indexing errors


597 prompts who_said reservados | 64 ventanas de texto reservado


cross-entropy de torch: 5.6791 nats/token


# Consigna I — exportar a ONNX

Un archivo ONNX es un **grafo de cómputo**: los operadores, su orden y los pesos, sin nada
de Python. `torch.onnx.export` lo produce *trazando* — corriendo tu modelo una vez con una
entrada de mentira y registrando las operaciones que efectivamente se ejecutaron.

El trazado es la razón por la que un export puede estar mal sin que nada falle. Cualquier
cosa decidida por un `if` de Python sobre el valor de un tensor queda congelada como lo que
haya pasado en esa única corrida.

**Qué implementar**

1. Un `nn.Module` que envuelva al modelo. `TinyGPT.forward` devuelve una tupla cuando
   `use_cache` o `return_weights` están activados, y un tensor pelado si no. El tracer
   necesita exactamente un camino con una sola salida, así que envolvelo y devolvé solo
   `logits`.
2. `torch.onnx.export(...)` con:
   - `input_names=["input_ids"]`, `output_names=["logits"]`;
   - `dynamic_axes` marcando **batch** y **secuencia** como dinámicos en los dos. Sin eso
     el grafo queda congelado en la forma de la entrada de mentira y solo acepta prompts
     de ese largo exacto;
   - `opset_version=17`, `do_constant_folding=True`.

**Dos trampas, las dos silenciosas**

- **`export()` deja el módulo en modo entrenamiento.** El dropout queda vivo del lado de
  PyTorch, tu chequeo de paridad marca ~4 en vez de ~1e-06, y te pasás la tarde culpando al
  exportador. Llamá a `.eval()` de nuevo después de exportar — el self-check lo verifica.
- **La paridad es la única prueba que tenés.** Un grafo trazado que corre no es un grafo
  correcto. Compará los logits contra PyTorch en *varios* largos de secuencia, no en uno.

In [4]:
class ExportWrapper(nn.Module):
    """Una entrada, una salida, un camino por el modelo -- lo que el tracer necesita."""

    def __init__(self, m: nn.Module):
        super().__init__()
        self.m = m

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # Sin use_cache ni return_weights, TinyGPT.forward devuelve solo los logits.
        return self.m(input_ids)


def export_onnx(m: nn.Module, path: str) -> str:
    """
    Trazá `m` a un grafo ONNX en `path`, con batch y secuencia dinámicos.

    Devuelve `path`. Acordate de lo que el trazado le hace a `m.training`.
    """
    m.eval()
    # Una entrada de mentira de un largo cualquiera: los ejes dinámicos hacen que no importe.
    dummy = torch.randint(0, m.config.vocab_size, (1, 8), dtype=torch.long)
    torch.onnx.export(
        ExportWrapper(m), (dummy,), path,
        input_names=["input_ids"], output_names=["logits"],
        dynamic_axes={"input_ids": {0: "batch", 1: "seq"},
                      "logits": {0: "batch", 1: "seq"}},
        opset_version=17, do_constant_folding=True,
    )
    # El wrapper nace en modo entrenamiento, y export() le restaura ese modo al terminar,
    # recursivamente: deja al modelo con el dropout vivo.
    m.eval()
    return path

## Self-check

In [5]:
def check_export():
    path = export_onnx(model, f"{ONNX_DIR}/tinygpt.onnx")
    assert os.path.exists(path), "no se escribió nada"

    assert not model.training, \
        "el modelo está en modo entrenamiento -- lo dejó export(); llamá a .eval()"

    sess = make_session(path)
    names = [i.name for i in sess.get_inputs()]
    assert names == ["input_ids"], f"las entradas son {names}"

    worst = 0.0
    for T in (1, 8, 17, config.block_size):
        x = torch.randint(0, config.vocab_size, (1, T), dtype=torch.long)
        with torch.no_grad():
            ref = model(x).numpy()
        got = sess.run(None, {"input_ids": x.numpy()})[0]
        assert got.shape == ref.shape, f"T={T}: {got.shape} != {ref.shape}"
        worst = max(worst, float(np.abs(ref - got).max()))
    assert worst < 1e-4, f"los logits difieren en {worst:.2e} -- el grafo no es el modelo"

    # Un eje de batch dinámico es lo que después le permite a un server batchear requests.
    x = torch.randint(0, config.vocab_size, (4, 8), dtype=torch.long)
    assert sess.run(None, {"input_ids": x.numpy()})[0].shape[0] == 4, \
        "el eje de batch quedó congelado -- revisá dynamic_axes"

    print(f"paridad    max |torch - onnx| = {worst:.2e}  (seq 1/8/17/{config.block_size})")
    print(f"muestra    {greedy_onnx(sess, DEMO_PROMPTS[1])!r}")
    print("todos los checks pasaron\n")
    # El mismo lector de recién -- ahora apuntado a un transformer.
    show_graph(path, max_nodes=6)
    return sess


fp32_sess = check_export()

/Users/fermin/Desktop/UBA/PLNII/tps-plnii/TP3/tinygpt.py:311: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert past_len + T <= self.config.block_size, (


paridad    max |torch - onnx| = 1.05e-05  (seq 1/8/17/32)
muestra    'DUKE VINCENTIO'
todos los checks pasaron

./onnx/tinygpt.onnx  |  221 nodos  |  30 tensores de pesos (26.1 MB)
  in   input_ids: int64[batch,seq]
  out  logits: float[batch,seq,50261]
  ops  {'Constant': 78, 'Unsqueeze': 25, 'Add': 15, 'Gather': 13, 'MatMul': 13, 'Shape': 11, 'Slice': 10, 'Transpose': 10}
    m.blocks.1.attn.tril           = Identity(*m.blocks.0.attn.tril)
    /m/Shape_output_0              = Shape(input_ids)
    /m/Constant_output_0           = Constant()
    /m/Gather_output_0             = Gather(/m/Shape_output_0, /m/Constant_output_0)
    /m/token_emb/Gather_output_0   = Gather(*m.token_emb.weight, input_ids)
    /m/Constant_1_output_0         = Constant()
    ... 215 más    (* = un peso guardado en el archivo)


## Mirá lo que acabás de fabricar

`check_export` imprimió la firma del grafo y su mezcla de operadores. Ahora abrí el archivo.

[**Netron**](https://netron.app) es el visor estándar para este formato, y la celda de abajo
lo embebe en el notebook si lo tenés instalado (`pip install netron`). Si no, abrí
[netron.app](https://netron.app) y arrastrá `onnx/tinygpt.onnx` sobre la página — se
renderiza localmente en tu browser, el archivo no se sube a ningún lado.

Cosas que vale la pena encontrar, porque son el punto del ejercicio:

- **Un `Gather` al principio y un `MatMul` de 50.261 de ancho al final** — el embedding y
  la cabeza de salida, las dos matrices que el TP-2 midió como el 98% de este modelo.
- **Tu máscara de atención de `block_size`, horneada como constante.** Es un `Slice` sobre
  un `tril` guardado, no código.
- **Nada de Python, ni clases, ni `forward`.** `TinyGPT` desapareció; lo que queda es
  aritmética.
- **Dos bloques que se ven idénticos.** Tu `n_layer = 2`, desenrollado — el loop que los
  escribió no existe en el archivo.

Encontrá un operador que vos no escribiste y explicá de dónde salió. El trazado agrega
nodos `Shape`, `Unsqueeze` y `Constant` por cada cuenta de formas que tu `forward` hizo a
nivel de Python, que es la razón por la que salen 221 nodos de un modelo que podés leer en
un minuto.

In [6]:
def view_graph(path: str, height: int = 600):
    """Embebe Netron si está instalado; si no, dice cómo ver el archivo a mano."""
    try:
        import netron
    except ImportError:
        print(f"netron no está instalado. O bien:\n"
              f"  pip install netron      (y volvé a correr esta celda)\n"
              f"  o abrí https://netron.app y arrastrá {path}")
        return show_graph(path, max_nodes=12)

    from IPython.display import IFrame
    host, port = netron.start(path, address=("localhost", 0), browse=False)
    print(f"sirviendo {path} en http://{host}:{port}   (netron.stop() para cerrarlo)")
    return IFrame(f"http://{host}:{port}", width="100%", height=height)


view_graph(f"{ONNX_DIR}/tinygpt.onnx")

netron no está instalado. O bien:
  pip install netron      (y volvé a correr esta celda)
  o abrí https://netron.app y arrastrá ./onnx/tinygpt.onnx
./onnx/tinygpt.onnx  |  221 nodos  |  30 tensores de pesos (26.1 MB)
  in   input_ids: int64[batch,seq]
  out  logits: float[batch,seq,50261]
  ops  {'Constant': 78, 'Unsqueeze': 25, 'Add': 15, 'Gather': 13, 'MatMul': 13, 'Shape': 11, 'Slice': 10, 'Transpose': 10}
    m.blocks.1.attn.tril           = Identity(*m.blocks.0.attn.tril)
    /m/Shape_output_0              = Shape(input_ids)
    /m/Constant_output_0           = Constant()
    /m/Gather_output_0             = Gather(/m/Shape_output_0, /m/Constant_output_0)
    /m/token_emb/Gather_output_0   = Gather(*m.token_emb.weight, input_ids)
    /m/Constant_1_output_0         = Constant()
    /m/Cast_output_0               = Cast(/m/Gather_output_0)
    /m/Constant_2_output_0         = Constant()
    /m/Range_output_0              = Range(/m/Constant_1_output_0, /m/Cast_output_0, /m/Constan)

## Informe — Consigna I

**Versiones.** Corrí con torch 2.2.2, onnx 1.23.0 y onnxruntime 1.23.2. El notebook se escribió contra torch 2.7.1, onnx 1.22.0 y onnxruntime 1.30.0; torch 2.2.2 es el último que hay para Mac con Intel. En esta versión `torch.onnx.export` usa el exportador por trazado (TorchScript). Los dos exports pasaron la paridad sin cambios: 1,05e-05 en la Consigna I y 7,63e-06 en la II.

**La trampa del modo entrenamiento existe y tiene una causa concreta.** `ExportWrapper` es un módulo nuevo, y un `nn.Module` nace en modo entrenamiento. `export()` pone el modelo en `eval` para trazar y al terminar le restaura el modo original al wrapper, de forma recursiva. Así el TinyGPT de adentro queda en `train`, con el dropout activo. Lo verifiqué antes de agregar el `m.eval()` final: `model.training` daba `True` después de exportar.

**Lo que hay en el grafo** (221 nodos, 26,1 MB):
- **Las dos matrices grandes.** `Gather(*m.token_emb.weight, input_ids)` es el embedding, y el `MatMul` contra `onnx::MatMul_318` de forma [64, 50261] es la cabeza. Cada una ocupa 12,9 MB: entre las dos son 25,7 de los 26,1 MB del archivo.
- **13 `MatMul`.** Son 2 bloques × 6 (qkv, q·kᵀ, pesos·v, proj y las dos capas de la MLP), más la cabeza. Las dos capas están desenrolladas y son idénticas.
- **5 `LayerNormalization`.** Dos por bloque, más `ln_f`.
- **Ningún `Dropout`.** En modo eval el exportador los elimina.

**Un operador que no escribí: `m.blocks.1.attn.tril = Identity(*m.blocks.0.attn.tril)`.** Cada capa de atención registra su propio buffer `tril` de 32×32. Como los dos son idénticos, el exportador guarda uno solo como peso y define el segundo como un `Identity` del primero: deduplicó una constante.

**De dónde salen los otros nodos.** La cadena del principio, `Shape(input_ids)` → `Gather` → `Cast` → `Range` → `Gather(*m.pos_emb.weight, …)`, es `torch.arange(past_len, past_len + T)`. El tracer no congeló `T` como un número: registró la lectura de la forma (`Shape` + `Gather` del eje 1). Gracias a eso el eje de secuencia es dinámico. Lo mismo pasa con cada `view(B, T, n_head, head_dim)` y con el corte `tril[past:T_k, :T_k]`: cada entero de Python que salía de una forma ahora es un tensor que se calcula en el grafo. Esa es la fuente de los 78 `Constant`, los 25 `Unsqueeze`, los 11 `Shape` y los 8 `Concat`, más de la mitad de los nodos, en un modelo cuya aritmética es una docena de matmuls.

# Consigna II — exportar el KV-cache

El grafo que acabás de exportar vuelve a leer el prompt entero para cada token. La Consigna
III del TP-1 midió lo que ahorra un KV-cache; ahora tiene que sobrevivir el viaje a ONNX.

Un runtime no tiene objetos de Python donde guardar estado entre llamadas, así que el cache
no puede vivir en una lista al costado. **Pasa a ser entrada y salida del grafo**: quien
llama pasa el pasado y recibe el presente. Así exactamente se sirven GPT-2, Llama y
cualquier otro decoder exportado.

`TinyGPT` guarda un tensor apilado por capa, de forma
`(2, batch, n_head, past, head_dim)` — key y value apilados en el eje 0. Con `n_layer = 2`
eso son 2 tensores de entrada y 2 de salida, más `input_ids`.

**Qué implementar**

1. `KVWrapper`, cuyo `forward(input_ids, *past)` llama al modelo con
   `kv_cache=list(past), use_cache=True` y devuelve `(logits, *present)`.
2. `export_onnx_kv`, trazándolo con un cache real de un forward real, nombrando los
   tensores `past_0..` y `present_0..`, y marcando el **eje del largo del pasado como
   dinámico** en todos ellos.

**Acá la trampa es la máscara.** La atención corta `tril[past:T_k, :T_k]`, donde `past`
sale de la forma de un tensor. Si ese corte se traza como constante, el grafo funciona
perfecto en el largo con el que lo trazaste y está mal en todos los demás — por eso el
self-check corre varios largos de pasado, no uno.

**Pista.** Armá el cache con el que vas a trazar así:

```python
with torch.no_grad():
    _, cache = m(some_ids, use_cache=True)
```

In [7]:
class KVWrapper(nn.Module):
    """Cache que entra, cache que sale -- un runtime no guarda estado entre llamadas."""

    def __init__(self, m: nn.Module):
        super().__init__()
        self.m = m

    def forward(self, input_ids: torch.Tensor, *past: torch.Tensor):
        logits, present = self.m(input_ids, kv_cache=list(past), use_cache=True)
        return (logits, *present)


def export_onnx_kv(m: nn.Module, path: str) -> str:
    """
    Exportá un grafo de un paso que recibe el cache pasado y devuelve el presente.

    Entradas: input_ids, past_0 .. past_{n_layer-1}
    Salidas:  logits,    present_0 .. present_{n_layer-1}
    """
    m.eval()
    n = m.config.n_layer
    # Un cache real, de un forward real, con un pasado de largo distinto de 0 y de 1.
    with torch.no_grad():
        _, cache = m(torch.randint(0, m.config.vocab_size, (1, 5), dtype=torch.long),
                     use_cache=True)
    step = torch.randint(0, m.config.vocab_size, (1, 1), dtype=torch.long)

    past_names = [f"past_{i}" for i in range(n)]
    present_names = [f"present_{i}" for i in range(n)]
    # Cache: (2, batch, n_head, largo, head_dim). El eje 3 crece en cada paso.
    dynamic = {"input_ids": {0: "batch", 1: "seq"}, "logits": {0: "batch", 1: "seq"}}
    dynamic.update({p: {1: "batch", 3: "past_len"} for p in past_names})
    dynamic.update({p: {1: "batch", 3: "total_len"} for p in present_names})

    torch.onnx.export(
        KVWrapper(m), (step, *cache), path,
        input_names=["input_ids"] + past_names, output_names=["logits"] + present_names,
        dynamic_axes=dynamic, opset_version=17, do_constant_folding=True,
    )
    m.eval()
    return path

## Self-check

In [8]:
def empty_cache(batch: int = 1) -> List[np.ndarray]:
    """Lo que alimentás en el primer paso: un cache con cero posiciones de pasado."""
    head_dim = config.n_embd // config.n_head
    return [np.zeros((2, batch, config.n_head, 0, head_dim), dtype=np.float32)
            for _ in range(config.n_layer)]


def check_kv_export():
    path = export_onnx_kv(model, f"{ONNX_DIR}/tinygpt.kv.onnx")
    assert not model.training, "export() volvió a dejar el modelo en modo entrenamiento"

    sess = make_session(path)
    ins = [i.name for i in sess.get_inputs()]
    outs = [o.name for o in sess.get_outputs()]
    assert ins == ["input_ids"] + [f"past_{i}" for i in range(config.n_layer)], ins
    assert outs == ["logits"] + [f"present_{i}" for i in range(config.n_layer)], outs

    # El punto del test: largos de pasado que el export nunca vio.
    worst = 0.0
    for past_len in (0, 3, 8, 20, config.block_size - 1):
        ids = torch.randint(0, config.vocab_size, (1, max(past_len, 1)), dtype=torch.long)
        if past_len == 0:
            cache = empty_cache()
        else:
            with torch.no_grad():
                _, c = model(ids, use_cache=True)
            cache = [t.numpy() for t in c]

        step = torch.randint(0, config.vocab_size, (1, 1), dtype=torch.long)
        feed = {"input_ids": step.numpy()}
        feed.update({f"past_{i}": c for i, c in enumerate(cache)})
        got = sess.run(None, feed)

        torch_cache = None if past_len == 0 else [torch.from_numpy(c) for c in cache]
        with torch.no_grad():
            ref, present = model(step, kv_cache=torch_cache, use_cache=True)
        worst = max(worst, float(np.abs(ref.numpy() - got[0]).max()))
        assert got[1].shape[3] == past_len + 1, \
            f"el cache presente mide {got[1].shape[3]}, se esperaba {past_len + 1}"
        assert np.abs(present[0].numpy() - got[1]).max() < 1e-4, "el cache devuelto difiere"

    assert worst < 1e-4, (f"los logits difieren en {worst:.2e} -- lo más probable es que la "
                          f"máscara se haya trazado como constante")
    print(f"paridad    max |torch - onnx| = {worst:.2e}  (past 0/3/8/20/{config.block_size - 1})")
    print("todos los checks pasaron\n")
    # Leé la firma: ahora el cache es parte del contrato, de entrada y de salida.
    show_graph(path, max_nodes=0)
    return sess


kv_sess = check_kv_export()

paridad    max |torch - onnx| = 7.63e-06  (past 0/3/8/20/31)
todos los checks pasaron

./onnx/tinygpt.kv.onnx  |  251 nodos  |  30 tensores de pesos (26.1 MB)
  in   input_ids: int64[batch,seq], past_0: float[2,batch,4,past_len,16], past_1: float[2,batch,4,past_len,16]
  out  logits: float[batch,seq,50261], present_0: float[2,batch,4,total_len,16], present_1: float[2,batch,4,total_len,16]
  ops  {'Constant': 88, 'Unsqueeze': 29, 'Add': 16, 'Gather': 14, 'Concat': 14, 'MatMul': 13, 'Shape': 12, 'Slice': 10}
    ... 251 más    (* = un peso guardado en el archivo)


# Consigna III — `generateV2`, portado al runtime

La Consigna I del TP-1 te hizo escribir `generateV2`: greedy, temperatura, top-k, top-p,
con un KV-cache opcional. Eso era PyTorch. **La decodificación no es parte del modelo** —
nunca lo fue, y por eso nada de eso aparece en el grafo que exportaste. Es un loop que
escribe quien llama, alrededor de lo que sea que calcule los logits.

Así que portalo. Las mismas perillas, la misma semántica que en el TP-1, contra una sesión
de ONNX en vez de un `nn.Module`. Dos diferencias, y son la lección:

- **No hay `model.forward`, solo un diccionario de entradas.** El truncado, el manejo del
  cache y la contabilidad de posiciones ahora son tuyos.
- **El cache no es estado escondido.** Con `use_cache=True` manejás el grafo de la Consigna
  II: alimentás `past_*`, leés `present_*`, y los devolvés en el paso siguiente.

**Qué implementar**

1. `sample_next(logits, ...)` — las reglas de decodificación, como función pura sobre una
   fila de logits. Greedy cuando `do_sample=False`; si no, temperatura, después top-k,
   después top-p, después samplear. `suppress` lista ids de tokens que nunca pueden
   elegirse (la Parte 3 lo necesita para el andamiaje del chat; la Consigna V del TP-2
   chocó con el mismo problema).
2. `stream_tokens(sess, prompt, ...)` — un **generador** que va entregando ids de tokens
   hasta `<|end|>` o `max_new_tokens`. `use_cache=False` maneja el grafo de la Consigna I,
   `use_cache=True` el de la Consigna II.

Un generador y no un string, porque la Parte 3 necesita los tokens a medida que aparecen.
`generate_onnx`, abajo, lo vuelve a envolver en texto y está dado.

**Detalles que importan**

- Ordená las perillas como en el TP-1: la temperatura escala los logits, después top-k
  trunca, después top-p trunca, después sampleás. Aplicar top-p antes de la temperatura da
  otra distribución.
- `top_p` se queda con el **conjunto más chico cuya probabilidad acumulada llega a p**, así
  que `top_p` cerca de 0 es prácticamente greedy, y `top_p=1.0` no trunca nada.
- Con el cache, después de la primera llamada solo entra el token **nuevo** como
  `input_ids`, y el cache hay que recortarlo en `block_size` — el grafo lo verifica.
- Llamá a `run_session(sess, feed)` en vez de `sess.run(...)`, para que la Consigna V pueda
  medir cuánto del tiempo fue realmente el engine.
- Sembrá con `np.random.default_rng(seed)` para poder reproducir una corrida sampleada.

In [9]:
def sample_next(logits: np.ndarray, do_sample: bool = True,
                temperature: float = 1.0, top_k: Optional[int] = None,
                top_p: Optional[float] = None, suppress: Sequence[int] = (),
                rng: Optional[np.random.Generator] = None) -> int:
    """
    Una fila de logits -> un id de token.

    do_sample=False -> arg-max, y todas las otras perillas se ignoran (como en el TP-1).
    Orden: temperatura, después top-k, después top-p, después samplear.
    """
    logits = np.array(logits, dtype=np.float64)          # copia: no tocar la salida del grafo
    if len(suppress):
        logits[list(suppress)] = -np.inf                  # vale también para greedy
    if not do_sample:
        return int(np.argmax(logits))

    rng = rng if rng is not None else np.random.default_rng()
    logits = logits / max(temperature, 1e-8)              # 1) temperatura

    if top_k is not None and 0 < top_k < logits.size:     # 2) top-k
        kth = np.partition(logits, -top_k)[-top_k]
        logits[logits < kth] = -np.inf

    probs = np.exp(logits - logits.max())
    probs /= probs.sum()

    if top_p is not None and top_p < 1.0:                 # 3) top-p: el conjunto más chico
        order = np.argsort(-probs)                        #    cuya masa acumulada llega a p
        keep = min(int(np.searchsorted(np.cumsum(probs[order]), top_p)) + 1, probs.size)
        kept = np.zeros(probs.size, dtype=bool)
        kept[order[:keep]] = True
        probs = np.where(kept, probs, 0.0)
        probs /= probs.sum()

    return int(rng.choice(probs.size, p=probs))           # 4) samplear


def stream_tokens(sess: ort.InferenceSession, prompt: str, max_new_tokens: int = 24,
                  use_cache: bool = True, do_sample: bool = True,
                  temperature: float = 1.0, top_k: Optional[int] = None,
                  top_p: Optional[float] = None, suppress: Sequence[int] = (),
                  seed: Optional[int] = None, min_new_tokens: int = 0):
    """
    Va entregando ids de tokens generados, hasta END_ID o max_new_tokens.

    use_cache=False -> `sess` es el grafo de la Consigna I (solo input_ids).
    use_cache=True  -> `sess` es el de la Consigna II (past_* adentro, present_* afuera).
    min_new_tokens  -> <|end|> no se puede elegir antes de ese paso (lo usa el server de la
                       Consigna VI; en 0, el default, no cambia nada).
    """
    rng = np.random.default_rng(seed)
    knobs = dict(do_sample=do_sample, temperature=temperature, top_k=top_k, top_p=top_p,
                 rng=rng)
    early = list(suppress) + [END_ID]
    sup = lambda step: early if step < min_new_tokens else suppress
    ids = tokenizer.encode(prompt)
    block = config.block_size

    if not use_cache:
        for step in range(max_new_tokens):
            x = np.array([ids[-block:]], dtype=np.int64)        # la ventana entera, cada vez
            nxt = sample_next(run_session(sess, {"input_ids": x})[0][0, -1],
                              suppress=sup(step), **knobs)
            if nxt == END_ID:
                return
            yield nxt
            ids.append(nxt)
        return

    # Con cache: el primer paso lee el prompt entero contra un cache vacío; después entra
    # solo el token nuevo, y el cache que devuelve el grafo vuelve a entrar como pasado.
    past = empty_cache()
    x = np.array([ids[-block:]], dtype=np.int64)
    for step in range(max_new_tokens):
        feed = {"input_ids": x}
        feed.update({f"past_{i}": p for i, p in enumerate(past)})
        logits, *present = run_session(sess, feed)
        nxt = sample_next(logits[0, -1], suppress=sup(step), **knobs)
        if nxt == END_ID:
            return
        yield nxt
        # Pasado + el token nuevo tiene que entrar en block_size: se queda con los últimos
        # block_size - 1, igual que trim_kv_cache en el TP-1.
        past = [p[:, :, :, -(block - 1):, :] for p in present]
        x = np.array([[nxt]], dtype=np.int64)


def generate_onnx(sess: ort.InferenceSession, prompt: str, **kw) -> str:
    """Dado: el wrapper que devuelve texto, para que todos coincidan en el contrato."""
    return tokenizer.decode(list(stream_tokens(sess, prompt, **kw)))

## Self-check

In [10]:
def check_generate():
    prompt = DEMO_PROMPTS[1]
    greedy = dict(do_sample=False, max_new_tokens=14)

    # 1. el camino sin cache tiene que reproducir exactamente el loop de referencia
    mine = generate_onnx(fp32_sess, prompt, use_cache=False, **greedy)
    assert mine == greedy_onnx(fp32_sess, prompt), f"{mine!r} != referencia"

    # 2. y el camino con cache tiene que coincidir -- mismos pesos, mismas decisiones
    cached = generate_onnx(kv_sess, prompt, use_cache=True, **greedy)
    assert cached == mine, f"el cache cambia la salida: {cached!r} != {mine!r}"

    # 3. do_sample=False ignora todas las demás perillas
    loud = generate_onnx(kv_sess, prompt, temperature=5.0, top_k=999, top_p=0.3, **greedy)
    assert loud == mine, "do_sample=False tiene que ignorar temperature/top_k/top_p"

    # 4. los valores degenerados de cada perilla colapsan a greedy
    assert generate_onnx(kv_sess, prompt, max_new_tokens=14, top_k=1, seed=0) == mine, \
        "top_k=1 tiene que ser greedy"
    assert generate_onnx(kv_sess, prompt, max_new_tokens=14, top_p=1e-6, seed=0) == mine, \
        "top_p -> 0 se queda solo con el token más probable"

    # 5. el sampleo realmente samplea, y una semilla lo hace reproducible
    draws = {generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=s)
             for s in range(8)}
    assert len(draws) > 1, "temperature=2.0 dio una sola salida -- no está sampleando"
    a = generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=7)
    b = generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=7)
    assert a == b, "la misma semilla tiene que dar el mismo texto"

    # 6. los ids suprimidos no se emiten nunca, por más caliente que sea el sampleo
    banned = [int(np.array(tokenizer.encode(mine))[0])] if mine else []
    if banned:
        for seed in range(4):
            ids = list(stream_tokens(kv_sess, prompt, max_new_tokens=14, temperature=2.0,
                                     suppress=banned, seed=seed))
            assert banned[0] not in ids, "se sampleó un id suprimido"

    # 7. sample_next es una función pura -- probala sin modelo ninguno
    logits = np.array([0.0, 10.0, 5.0, -1.0])
    assert sample_next(logits, do_sample=False) == 1
    assert sample_next(logits, do_sample=False, suppress=[1]) == 2
    rng = np.random.default_rng(0)
    assert sample_next(logits, top_k=1, rng=rng) == 1

    print(f"greedy       {mine!r}")
    print(f"con cache    {cached!r}  (idéntico)")
    print(f"temp 2.0     {sorted(draws)[0]!r}")
    print(f"salidas distintas con 8 semillas: {len(draws)}")
    print("todos los checks pasaron")


check_generate()

greedy       'DUKE VINCENTIO'
con cache    'DUKE VINCENTIO'  (idéntico)
temp 2.0     ' 2017 waternuil seeks GRE backon Orа hint kn clin TOD'
salidas distintas con 8 semillas: 8
todos los checks pasaron


# Consigna IV — INT8, y lo que cuesta

La cuantización dinámica guarda los pesos como enteros de 8 bits y los descuantiza al
vuelo. No necesita datos de calibración y es una sola llamada:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QUInt8)
```

**El test de la Consigna I ya no aplica.** La cuantización cambia la aritmética a
propósito. La paridad de logits contra PyTorch va a ser grande — del orden de 1e-01 en vez
de 1e-06 — y eso no es un bug para arreglar. Significa que cambió la pregunta: ya no es
*¿es el mismo cálculo?* sino **¿es el mismo modelo, en comportamiento?**

Así que medí comportamiento, en dos ejes:

- **calidad** — cross-entropy sobre texto reservado, fp32 contra INT8. ¿El modelo empeoró
  modelando lenguaje?
- **fidelidad** — coincidencia con fp32. La **coincidencia top-1** es con qué frecuencia
  los dos eligen el mismo token siguiente; la **coincidencia de respuestas** es con qué
  frecuencia un decode greedy entero sale idéntico carácter por carácter. La segunda es
  mucho más dura, porque un solo token distinto al principio cambia todo lo que sigue.

Reportá juntos tamaño, paridad, cross-entropy y las dos coincidencias. Uno de esos números
va a parecer alarmante y el resto no; tu informe tiene que decir cuál creés y por qué.

El self-check de abajo verifica la *medición*, no el modelo: una comparación que no puede
reconocer una sesión como idéntica a sí misma volvería ruido todos estos números.

In [11]:
def quantize_int8(fp32_path: str, int8_path: str) -> str:
    """Cuantización dinámica a INT8 de un archivo ONNX existente. Devuelve int8_path."""
    from onnxruntime.quantization import QuantType, quantize_dynamic
    quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QUInt8)
    return int8_path


def _next_logits(sess: ort.InferenceSession, prompt: str) -> np.ndarray:
    x = np.array([tokenizer.encode(prompt)[-config.block_size:]], dtype=np.int64)
    return sess.run(None, {"input_ids": x})[0][0, -1]


def top1_agreement(sess_a: ort.InferenceSession, sess_b: ort.InferenceSession,
                   n: int = 60) -> float:
    """
    Proporción de prompts donde las dos sesiones predicen el mismo token siguiente.

    Las dos sesiones toman solo `input_ids`. Un forward cada una, sin decodificar.
    """
    prompts = [p for p, _ in EVAL_SET[:n]]
    same = sum(int(_next_logits(sess_a, p).argmax()) == int(_next_logits(sess_b, p).argmax())
               for p in prompts)
    return same / len(prompts)


def reply_agreement(sess_a: ort.InferenceSession, sess_b: ort.InferenceSession,
                    n: int = 40) -> float:
    """Proporción de prompts donde los dos decodes greedy son strings idénticos."""
    prompts = [p for p, _ in EVAL_SET[:n]]
    return sum(greedy_onnx(sess_a, p) == greedy_onnx(sess_b, p) for p in prompts) / len(prompts)


def position_agreement(sess_a: ort.InferenceSession, sess_b: ort.InferenceSession) -> float:
    """Extra: top-1 en cada posición del texto reservado (64 x 32 decisiones, no 60)."""
    same = total = 0
    for x, _ in LM_WINDOWS:
        a = sess_a.run(None, {"input_ids": x})[0][0].argmax(-1)
        b = sess_b.run(None, {"input_ids": x})[0][0].argmax(-1)
        same, total = same + int((a == b).sum()), total + a.size
    return same / total


def parity(sess: ort.InferenceSession) -> float:
    """max |torch - onnx| sobre las primeras 8 ventanas de texto reservado."""
    return max(float(np.abs(torch_logits(x) - sess.run(None, {"input_ids": x})[0][0]).max())
               for x, _ in LM_WINDOWS[:8])


int8_path = quantize_int8(f"{ONNX_DIR}/tinygpt.onnx", f"{ONNX_DIR}/tinygpt.int8.onnx")
int8_sess = make_session(int8_path)

torch_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
ce = {"torch": cross_entropy(torch_logits),
      "fp32": cross_entropy(onnx_logits(fp32_sess)),
      "int8": cross_entropy(onnx_logits(int8_sess))}

print(f"{'artefacto':<14} | {'tamaño MB':>9} | {'max |torch-x|':>13} | {'cross-ent.':>10} | "
      f"{'top-1':>6} | {'respuestas':>10} | {'top-1/pos.':>10}")
print("-" * 92)
print(f"{'pytorch fp32':<14} | {torch_mb:>9.1f} | {'—':>13} | {ce['torch']:>10.4f} | "
      f"{'—':>6} | {'—':>10} | {'—':>10}")
print(f"{'onnx fp32':<14} | {os.path.getsize(f'{ONNX_DIR}/tinygpt.onnx') / 1e6:>9.1f} | "
      f"{parity(fp32_sess):>13.2e} | {ce['fp32']:>10.4f} | {'ref.':>6} | {'ref.':>10} | {'ref.':>10}")
print(f"{'onnx int8':<14} | {os.path.getsize(int8_path) / 1e6:>9.1f} | "
      f"{parity(int8_sess):>13.2e} | {ce['int8']:>10.4f} | "
      f"{top1_agreement(fp32_sess, int8_sess):>6.1%} | "
      f"{reply_agreement(fp32_sess, int8_sess):>10.1%} | "
      f"{position_agreement(fp32_sess, int8_sess):>10.1%}")
print("\ncoincidencias contra onnx fp32: top-1 sobre 60 prompts who_said, respuestas greedy "
      "sobre 40,\ny top-1 en cada posición de las 64 ventanas de texto reservado.")

artefacto      | tamaño MB | max |torch-x| | cross-ent. |  top-1 | respuestas | top-1/pos.
--------------------------------------------------------------------------------------------
pytorch fp32   |      26.1 |             — |     5.6791 |      — |          — |          —
onnx fp32      |      26.2 |      1.14e-05 |     5.6791 |   ref. |       ref. |       ref.


onnx int8      |       6.6 |      3.78e-01 |     5.6809 | 100.0% |     100.0% |      98.5%

coincidencias contra onnx fp32: top-1 sobre 60 prompts who_said, respuestas greedy sobre 40,
y top-1 en cada posición de las 64 ventanas de texto reservado.


## Self-check

In [12]:
def check_quantized():
    # Una función de comparación que no se reconoce a sí misma como idéntica está rota, y
    # todos los números de la tabla de arriba serían ruido.
    assert top1_agreement(fp32_sess, fp32_sess) == 1.0, \
        "una sesión tiene que coincidir consigo misma el 100% de las veces"
    assert reply_agreement(fp32_sess, fp32_sess) == 1.0, \
        "una sesión tiene que reproducir sus propias respuestas exactamente"

    top1 = top1_agreement(fp32_sess, int8_sess)
    replies = reply_agreement(fp32_sess, int8_sess)
    assert 0.0 <= top1 <= 1.0 and 0.0 <= replies <= 1.0, "la coincidencia es una proporción"
    assert replies <= top1 + 1e-9, \
        "las respuestas enteras no pueden coincidir más seguido que los tokens sueltos"

    # INT8 perturba los pesos; no reemplaza el modelo. Un número muy bajo acá es un bug en
    # la comparación o la sesión equivocada, no daño de la cuantización.
    assert top1 > 0.5, \
        f"una coincidencia top-1 de {top1:.1%} es muy baja para ser solo cuantización"
    drift = cross_entropy(onnx_logits(int8_sess)) - cross_entropy(onnx_logits(fp32_sess))
    assert abs(drift) < 0.5, \
        f"la cross-entropy se movió {drift:+.3f} nats -- eso no es redondeo"

    assert os.path.getsize(int8_path) < 0.6 * os.path.getsize(f"{ONNX_DIR}/tinygpt.onnx"), \
        "el archivo int8 no es realmente más chico -- ¿corrió la cuantización?"

    print(f"coincidencia consigo misma   100.0%  (la medición funciona)")
    print(f"fp32 vs int8     top-1 {top1:.1%} | respuestas {replies:.1%} | "
          f"cross-entropy {drift:+.4f}")
    print("todos los checks pasaron")


check_quantized()

coincidencia consigo misma   100.0%  (la medición funciona)
fp32 vs int8     top-1 100.0% | respuestas 100.0% | cross-entropy +0.0018
todos los checks pasaron


## Informe — Consigna IV

| artefacto | tamaño | max \|torch − x\| | cross-entropy | top-1 (60 prompts) | respuestas (40) | top-1 por posición (2.048) |
|---|---:|---:|---:|---:|---:|---:|
| PyTorch fp32 | 26,1 MB | — | 5,6791 | — | — | — |
| ONNX fp32 | 26,2 MB | 1,14e-05 | 5,6791 | ref. | ref. | ref. |
| ONNX INT8 | **6,6 MB** | **3,78e-01** | 5,6809 | 100 % | 100 % | 98,5 % |

**El número que parece alarmante es la paridad: 3,78e-01, cuatro órdenes de magnitud peor que fp32.** No lo creo como medida del daño, porque mide otra cosa: la peor diferencia absoluta en cualquiera de los 50.261 logits de cualquier posición. Casi todos esos logits son de tokens con probabilidad despreciable, y moverlos un 0,4 no cambia ninguna decisión.

**Los números que sí creo son los de comportamiento:**
- **Calidad:** la cross-entropy sube 0,0018 nats, un 0,03 %.
- **Fidelidad:** el siguiente token coincide en los 60 prompts reservados y en el 98,5 % de 2.048 posiciones de texto. El decode greedy completo coincide carácter por carácter en las 40 respuestas.

El top-1 cambia en un 1,5 % de las posiciones y aun así la cross-entropy casi no se mueve. Eso sugiere (no lo medí) que esos cambios son casi empates entre los dos primeros candidatos.

La cuantización también cambió el grafo. Las 9 multiplicaciones contra pesos (4 por bloque, más la cabeza) pasaron a ser `DynamicQuantizeLinear` + `MatMulInteger` + `Cast` + `Mul`. El embedding se guarda en `uint8` y se descuantiza después del `Gather`. Las dos matmuls de la atención, q·kᵀ y pesos·v, siguen en fp32 porque son entre activaciones, no contra pesos.

# Consigna V — benchmark

Cuatro formas de producir los mismos tokens: PyTorch, ONNX sin cache, ONNX con cache, e
INT8. Tu `stream_tokens` ya maneja tres de ellas — corrélo greedy, así la comparación es
sobre el runtime y no sobre la suerte del sampleo.

Medí dos cosas por corrida, no una:

- **tiempo de reloj** del loop entero;
- **tiempo de engine** — `ENGINE["seconds"]`, que `run_session` viene acumulando por vos.
  Reseteálo antes de cada corrida y leelo después.

La diferencia entre los dos es Python: tokenizar, cortar, armar el diccionario de entradas,
samplear. Con 6.5M de parámetros esa diferencia no es un error de redondeo, y notarlo es el
punto del ejercicio. También es la razón por la que los servers de verdad no se escriben
como un `for` de Python.

**Reglas para un benchmark que valga la pena reportar:** calentá primero (la primera
llamada a `run` construye los kernels), usá el mismo prompt y la misma cantidad de
tokens en todos lados, y repetí lo suficiente como para ver el ruido. Aclará en qué
máquina corriste.

In [13]:
import platform
import subprocess

N_TOKENS, WARMUP, REPEATS = 32, 3, 3


def timed(decode, n_tokens: int = N_TOKENS) -> Tuple[float, float]:
    """
    Calentá, después corré `decode(n_tokens)` y devolvé (segundos de reloj, de engine).

    El tiempo de engine sale de ENGINE["seconds"] -- reseteálo justo antes de medir.
    Quedate con la mejor de REPEATS corridas: el mínimo es la estimación menos ruidosa.
    """
    for _ in range(WARMUP):
        decode(n_tokens)
    best = None
    for _ in range(REPEATS):
        ENGINE["seconds"] = 0.0
        t0 = time.perf_counter()
        decode(n_tokens)
        wall = time.perf_counter() - t0
        if best is None or wall < best[0]:
            best = (wall, ENGINE["seconds"])
    return best


def torch_decode(n_tokens: int) -> None:
    """Dado: el baseline de PyTorch, greedy, sin cache -- lo que ONNX tiene que ganarle."""
    ids = tokenizer.encode(DEMO_PROMPTS[0])
    for _ in range(n_tokens):
        x = torch.tensor(ids[-config.block_size:], dtype=torch.long)[None, :]
        t0 = time.perf_counter()
        with torch.no_grad():
            logits = model(x)
        ENGINE["seconds"] += time.perf_counter() - t0
        ids.append(int(logits[0, -1].argmax()))


def onnx_decode(sess: ort.InferenceSession, use_cache: bool):
    """Tu generador, greedy, y sin frenar antes: <|end|> suprimido, así son N tokens exactos."""
    def decode(n_tokens: int) -> None:
        ids = list(stream_tokens(sess, DEMO_PROMPTS[0], max_new_tokens=n_tokens,
                                 use_cache=use_cache, do_sample=False, suppress=[END_ID]))
        assert len(ids) == n_tokens
    return decode


try:
    cpu = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                         capture_output=True, text=True).stdout.strip()
except Exception:
    cpu = ""
print(f"máquina  {cpu or platform.processor()} | {platform.system()} {platform.machine()} | "
      f"{os.cpu_count()} hilos | torch usa {torch.get_num_threads()} | "
      f"carga promedio {os.getloadavg()[0]:.1f}")
print(f"{N_TOKENS} tokens por corrida, greedy, mismo prompt; mejor de {REPEATS} tras {WARMUP} "
      f"de calentamiento\n")

# Extra, para la pregunta 5: el grafo con cache también cuantizado.
int8_kv_sess = make_session(quantize_int8(f"{ONNX_DIR}/tinygpt.kv.onnx",
                                          f"{ONNX_DIR}/tinygpt.int8.kv.onnx"))
same = all(generate_onnx(int8_kv_sess, p, do_sample=False) == generate_onnx(kv_sess, p, do_sample=False)
           for p in DEMO_PROMPTS)
print(f"extra: int8 con cache -- greedy idéntico a fp32 con cache en los DEMO_PROMPTS: {same}\n")

runs = [("torch cpu", torch_decode),
        ("onnx fp32 (sin cache)", onnx_decode(fp32_sess, use_cache=False)),
        ("onnx fp32 (kv cache)", onnx_decode(kv_sess, use_cache=True)),
        ("onnx int8 (sin cache)", onnx_decode(int8_sess, use_cache=False)),
        ("onnx int8 (kv cache) *", onnx_decode(int8_kv_sess, use_cache=True))]
bench = {name: timed(fn) for name, fn in runs}

base_wall = bench["torch cpu"][0]
print(f"{'corrida':<22} | {'ms/token':>8} | {'tok/s':>7} | {'engine':>7} | {'python':>7} | "
      f"{'vs torch':>8}")
print("-" * 74)
for name, (wall, engine) in bench.items():
    print(f"{name:<22} | {1e3 * wall / N_TOKENS:>8.3f} | {N_TOKENS / wall:>7.0f} | "
          f"{engine / wall:>7.0%} | {1 - engine / wall:>7.0%} | {base_wall / wall:>7.2f}x")
print("\n* extra, no pedida: la combinación de las dos optimizaciones (pregunta 5).")

máquina  Intel(R) Core(TM) i5-8257U CPU @ 1.40GHz | Darwin x86_64 | 8 hilos | torch usa 4 | carga promedio 5.4
32 tokens por corrida, greedy, mismo prompt; mejor de 3 tras 3 de calentamiento



extra: int8 con cache -- greedy idéntico a fp32 con cache en los DEMO_PROMPTS: False



corrida                | ms/token |   tok/s |  engine |  python | vs torch
--------------------------------------------------------------------------
torch cpu              |    2.188 |     457 |     90% |     10% |    1.00x
onnx fp32 (sin cache)  |    1.186 |     843 |     90% |     10% |    1.84x
onnx fp32 (kv cache)   |    0.613 |    1631 |     82% |     18% |    3.57x
onnx int8 (sin cache)  |    0.929 |    1076 |     89% |     11% |    2.35x
onnx int8 (kv cache) * |    0.421 |    2375 |     76% |     24% |    5.20x

* extra, no pedida: la combinación de las dos optimizaciones (pregunta 5).


# Preguntas

Respondelas con los números que acabás de producir. Sin volver a correr nada.

1. Tu modelo INT8 es unas **4x más chico** pero ni cerca de 4x más rápido. Explicá los dos
   números con el mismo hecho sobre esta arquitectura — el TP-2 lo midió, y es la razón por
   la que "parameter-efficient" era una pregunta real allá.
2. El grafo con KV-cache hace estrictamente menos aritmética por token que el que no lo
   tiene. ¿Ganó por el margen que esperabas? Sea cual sea la respuesta, explicala en
   términos de `block_size = 32` y de la proporción de Python en tu propia tabla.
3. La paridad de logits es el test correcto en la Consigna I y el equivocado en la Consigna
   IV. Decí qué mide realmente cada uno de tus tres tests — paridad, cross-entropy,
   coincidencia — y dame un cambio en un modelo que la coincidencia detectaría mientras la
   cross-entropy queda igual.
4. El trazado registra una sola ejecución. Nombrá un cambio en `TinyGPT.forward` que
   exportaría limpio, pasaría un chequeo de paridad de un solo largo, y aun así estaría mal
   en producción — y decí qué línea del self-check de la Consigna II lo atraparía.
5. Exportaste los mismos pesos tres veces. ¿Qué artefacto mandarías realmente a un
   teléfono, y cuál a un server con GPU? Defendé las dos elecciones con un número.

---

**Respuestas**

Benchmark corrido en un Intel Core i5-8257U (4 núcleos, 8 hilos, 1,4 GHz, AVX2 sin VNNI), con macOS y otros procesos andando (carga promedio 5,4). Entre corridas los tiempos variaron ~10 %; las proporciones se mantuvieron.

1. **Las dos cosas salen del mismo hecho: el embedding y la cabeza son el 98,4 % de los parámetros** (TP-2).
   - **Tamaño.** Las dos matrices de 50.261 × 64 pesan 12,9 MB cada una, 25,7 de los 26,2 MB del archivo. INT8 las lleva a 3,2 MB cada una, y el archivo queda en 6,6 MB: 3,97 veces más chico. El tamaño es proporcional a los parámetros, y casi todos los parámetros se cuantizan.
   - **Velocidad.** Sin cache, INT8 es apenas 1,28 veces más rápido (1,186 → 0,929 ms/token). La mitad de esos bytes es el embedding, que es un `Gather`: en cada paso lee 32 filas, unos pocos KB, sin aritmética. Achicarlo 4 veces no ahorra tiempo. La otra mitad es la cabeza, la única matmul grande. Pero su salida sigue en fp32: 32 × 50.261 logits por paso, que INT8 tiene que pasar por `Cast` y `Mul` para reescalar, además de cuantizar las activaciones en cada llamada (`DynamicQuantizeLinear`). En esta CPU, que no tiene instrucciones de producto punto int8 (VNNI), `MatMulInteger` no le saca mucha ventaja a una matmul fp32 con AVX2. Las capas donde INT8 suele acelerar, la atención y la MLP, son menos del 2 % de los parámetros.

   En resumen, el tamaño escala con los parámetros y el tiempo no.

2. **No ganó por el margen que uno esperaría si mira solo la aritmética.** Sin cache, cada paso recalcula la ventana entera de 32 posiciones; con cache calcula 1, unas 32 veces menos aritmética. El cache ganó 1,93 veces (1,186 → 0,613 ms/token).
   - **`block_size = 32` acota lo que el cache puede ahorrar.** Sin cache el costo por token no crece con el largo de la generación, porque la ventana nunca pasa de 32. En un modelo con 2.048 tokens de contexto la diferencia sería enorme.
   - **El tiempo del engine no escala con la cantidad de posiciones.** Pasa de 1,067 a 0,503 ms/token: la mitad, con 32 veces menos filas. Lo que domina el paso no depende de cuántas posiciones se calculan. Hay que leer los 12,9 MB de la cabeza, con 1 fila igual que con 32, y despachar 251 nodos de ONNX Runtime. Con cache la cabeza es un producto matriz-vector limitado por el ancho de banda de memoria; sin cache es una matmul que reusa cada peso 32 veces.
   - **Python es un costo fijo por token.** En la tabla, Python pasa del 10 % al 18 % del tiempo, pero en absoluto es lo mismo: 0,119 contra 0,110 ms/token. Es armar el diccionario de entradas, el arg-max sobre 50.261 logits y recortar el cache. Cuanto más rápido es el engine, más pesa esa parte fija (ley de Amdahl). Con INT8 y cache llega al 24 %.

3. **Cada test mide una cosa distinta:**
   - **Paridad:** si el grafo calcula *la misma función* que PyTorch, hasta el redondeo de float32. Es el test correcto para un export, donde cualquier diferencia de más de ~1e-5 es un bug. Para INT8 es el equivocado, porque ahí la aritmética cambia a propósito y la pregunta deja de ser "¿es la misma cuenta?".
   - **Cross-entropy:** la calidad como modelo de lenguaje, promediada sobre texto reservado. Sólo mira la probabilidad que el modelo le asigna al token correcto.
   - **Coincidencia:** la fidelidad de las *decisiones*. Mira si el orden de los candidatos cambia lo suficiente como para que el arg-max (o la respuesta entera) salga distinto.

   **Un cambio que la coincidencia detecta y la cross-entropy no:** mover masa de probabilidad entre tokens *incorrectos*. Si en una posición el token correcto tiene 0,10 y el modelo pasa de preferir `DUKE` (0,40) sobre `QUEEN` (0,35) a preferir `QUEEN` (0,40) sobre `DUKE` (0,35), la cross-entropy no cambia: sólo depende del 0,10. El decode greedy, en cambio, emite otro personaje. Lo mismo vale para cualquier perturbación que reordene casi empates sin tocar la probabilidad del token correcto.

4. **Un cambio en el `forward`: calcular `past` como un entero de Python.** Por ejemplo, `past = int(T_k - T)` en `MultiHeadAttention.forward`, o `past_len = int(kv_cache[0].shape[-2])` en `TinyGPT.forward`.
   - El tracer no puede seguir un `int()`, así que lo registra como la constante que valía durante el trazado: 5, el largo del cache con el que exporté.
   - La máscara `tril[5:5 + T, :T_k]` y las posiciones `arange(5, 5 + T)` quedan horneadas.
   - El export corre sin errores (a lo sumo una `TracerWarning`), y un chequeo de paridad con un pasado de largo 5 da ~1e-6. En producción, cada paso con otro largo de pasado usa la máscara y las posiciones equivocadas: números plausibles y texto sutilmente mal.

   Lo atrapa la línea `for past_len in (0, 3, 8, 20, config.block_size - 1):` de `check_kv_export`. Prueba largos de pasado que el export nunca vio, y después `assert worst < 1e-4` falla en todos.

5. **A un teléfono le mandaría el INT8, idealmente el grafo con cache cuantizado.** Pesa 6,6 MB en vez de 26,2: 4 veces menos descarga, almacenamiento y RAM. Cuesta +0,0018 nats de cross-entropy, con 100 % de coincidencia en el siguiente token. Con cache fue además el más rápido de la tabla extra: 0,421 ms/token, 5,2 veces PyTorch. En un teléfono lo que limita es la memoria y el ancho de banda, justo lo que INT8 ahorra. Hay que validarlo: en uno de los tres `DEMO_PROMPTS` difiere de fp32. Lo revisé aparte: en el `continue` frena un token más tarde, y en los 40 prompts `who_said` coincide en todos.

   **A un server con GPU, el grafo fp32 (o fp16) con KV-cache** (0,613 ms/token en CPU, 3,6 veces PyTorch).
   - Allá los 26,2 MB no importan: son menos del 0,1 % de una GPU de 24 GB.
   - La cuantización dinámica de ONNX Runtime (`MatMulInteger`, `DynamicQuantizeLinear`) está pensada para CPU; en GPU no tiene kernels buenos y puede terminar más lenta.
   - El cache es lo que escala con contextos largos y con batching.

# Consigna VI — un endpoint compatible con OpenAI

El grafo corre. Ahora dale la interfaz que el resto del mundo ya habla.

Todo modelo hosteado — OpenAI, y llama.cpp, vLLM, Ollama y TGI en sus modos de
compatibilidad — responde `POST /v1/chat/completions` con el mismo JSON. Hablá esa forma y
cualquier cliente existente funciona contra tu modelo sin tocarlo. Esa es toda la lección
de esta parte: **el contrato de la API es el límite del producto, no el modelo.**

El request que tenés que aceptar y la respuesta que tenés que devolver:

```json
{ "model": "tinygpt", "messages": [{"role": "user", "content": "continue: To be,"}],
  "max_tokens": 24, "temperature": 0.8 }
```

```json
{ "id": "chatcmpl-...", "object": "chat.completion", "created": 1700000000,
  "model": "tinygpt",
  "choices": [{ "index": 0,
                "message": {"role": "assistant", "content": "..."},
                "finish_reason": "stop" }],
  "usage": {"prompt_tokens": 12, "completion_tokens": 7, "total_tokens": 19} }
```

**Qué implementar**

1. `render_prompt(messages)` — los mensajes de OpenAI al template del TP-2. Un mensaje
   `system` no tiene dónde ir en un template que solo conoce `<|user|>` y `<|assistant|>`:
   decidí qué hacés con él y dejalo dicho en tu informe.
2. `decode_args(req)` — traducir las perillas del request a los argumentos que toma tu
   `stream_tokens` de la Consigna III. `temperature: 0` significa greedy, y hay que
   suprimir `SCAFFOLD_IDS`.
3. `generate_reply(req)` — correr la decodificación. Devolvé el texto **y** por qué
   frenaste, más las cuentas de tokens.
4. `build_response(req)` — armar el JSON de arriba.

Nada de código de decodificación nuevo: la Consigna III ya lo escribió. Esta consigna es la
capa de traducción entre una API pública y tu generador, que es casi todo lo que un stack
de serving hace.

La ruta se registra unas celdas más abajo, una sola vez, cuando ya estén escritas esta
consigna y la VII. Registrar dos veces el mismo path en FastAPI no reemplaza el handler —
gana el primero, en silencio — así que todo el ruteo vive en una única celda dada.

**Tres detalles que son, en realidad, la nota**

- **`finish_reason`** es `"stop"` cuando el modelo emitió `<|end|>` y `"length"` cuando
  llegó a `max_tokens`. Los clientes bifurcan según eso — así sabe quien llama si conviene
  pedir más.
- **`usage`** tiene que contar tokens de verdad: prompt, completion, y la suma.
- **Suprimí el andamiaje.** `SCAFFOLD_IDS` tiene `<|user|>`, `<|assistant|>` y `<|pad|>`.
  Poné esos logits en `-inf` antes de samplear, o van a aparecer *adentro* de las
  respuestas — el mismo bug del que avisaba la Consigna V del TP-2, ahora en producción.

In [14]:
class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class ChatRequest(BaseModel):
    model: str = "tinygpt"
    messages: List[Message]
    max_tokens: int = 24
    temperature: float = 0.8
    top_p: float = 1.0            # del contrato de OpenAI -- lo hace la Consigna III
    stream: bool = False          # Consigna VII


app = FastAPI(title="TinyGPT")

# El grafo que sirve: el de la Consigna II, fp32 con KV-cache.
SERVE_SESS = kv_sess


def render_prompt(messages: List[Message]) -> str:
    """Mensajes de OpenAI -> el chat template del TP-2, terminando en <|assistant|>."""
    parts = []
    for msg in messages:
        if msg.role == "system":
            # Se descarta: el modelo nunca vio un system prompt, y el TP-2 mostró que rutea
            # por la palabra de la tarea justo después de <|user|>. Ver las preguntas.
            continue
        if msg.role == "user":
            parts.append(f"{USER}{msg.content}{ASSISTANT}")
        else:
            # Una respuesta anterior, cerrada con <|end|> como en los ejemplos del TP-2.
            parts.append(f"{msg.content}{END}")
    prompt = "".join(parts)
    return prompt if prompt.endswith(ASSISTANT) else prompt + ASSISTANT


def decode_args(req: ChatRequest) -> Dict:
    """
    Los argumentos que tu `stream_tokens` de la Consigna III necesita para este request.

    temperature 0 significa greedy; el andamiaje no puede samplearse nunca.
    """
    greedy = req.temperature <= 0
    return dict(max_new_tokens=req.max_tokens, use_cache=True,
                do_sample=not greedy,
                temperature=1.0 if greedy else req.temperature,
                top_p=req.top_p if req.top_p < 1.0 else None,
                suppress=SCAFFOLD_IDS,
                # Nunca una respuesta vacía: <|end|> no puede ser el primer token (el min_tokens
                # de vLLM). Sin esto, greedy sobre un who_said corto devuelve "" -- ver Consigna VIII.
                min_new_tokens=1)


def generate_reply(req: ChatRequest) -> Tuple[str, str, int, int]:
    """Devuelve (texto, finish_reason, n_tokens_prompt, n_tokens_completion)."""
    prompt = render_prompt(req.messages)
    produced = list(stream_tokens(SERVE_SESS, prompt, **decode_args(req)))
    # Menos tokens que max_tokens solo puede querer decir que frenó en <|end|>.
    finish = "length" if len(produced) >= req.max_tokens else "stop"
    return tokenizer.decode(produced), finish, len(tokenizer.encode(prompt)), len(produced)


def build_response(req: ChatRequest) -> Dict:
    """La respuesta sin streaming documentada arriba."""
    text, finish, n_prompt, n_completion = generate_reply(req)
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{"index": 0,
                     "message": {"role": "assistant", "content": text},
                     "finish_reason": finish}],
        "usage": {"prompt_tokens": n_prompt, "completion_tokens": n_completion,
                  "total_tokens": n_prompt + n_completion},
    }

# Consigna VII — streaming

Tu endpoint hace esperar a quien llama hasta el último token antes de mostrarle el primero.
Con 24 tokens en una CPU eso es una pausa; con 500 tokens es una interfaz muerta. Todos los
productos de chat que usaste hacen streaming, y el mecanismo es más viejo y más simple que
el modelo: **server-sent events**, una respuesta HTTP larga que se escribe de a pedazos.

El formato es texto plano. Cada evento es una línea `data:` con un objeto JSON, separados
por una línea en blanco, y el stream termina con un centinela literal:

```
data: {"id":"chatcmpl-abc","object":"chat.completion.chunk","created":1700000000,"model":"tinygpt","choices":[{"index":0,"delta":{"role":"assistant"},"finish_reason":null}]}

data: {"id":"chatcmpl-abc", ... ,"choices":[{"index":0,"delta":{"content":"DUKE"},"finish_reason":null}]}

data: {"id":"chatcmpl-abc", ... ,"choices":[{"index":0,"delta":{},"finish_reason":"stop"}]}

data: [DONE]

```

Tres diferencias con la Consigna VI, y los clientes dependen de las tres:

- `object` es **`chat.completion.chunk`**, y cada choice lleva un **`delta`**, no un
  `message`. El primer delta anuncia el rol, los del medio llevan texto, el último está
  vacío y lleva `finish_reason`.
- El `id` es **el mismo en todos los chunks** de una misma respuesta.
- `usage` no está. No se conoce hasta el final, y OpenAI solo lo manda si se lo pedís con
  `stream_options`. Dejarlo afuera es el default compatible.

**Qué implementar**

`sse_events(req)`, un generador que va entregando esos strings `data: ...\n\n`.
Decodificá iterando tu `stream_tokens` de la Consigna III — un solo loop de decodificación,
dos presentaciones.

**La trampa: no decodifiques los tokens de a uno.** El BPE a nivel de bytes parte
caracteres entre tokens, así que `tokenizer.decode([one_id])` puede devolverte medio
carácter. Decodificá todo lo producido hasta ahora y emití solo la parte que todavía no
mandaste:

```python
text = tokenizer.decode(produced)
delta, sent = text[len(sent):], text
```

In [15]:
def sse_chunk(chunk_id: str, model: str, delta: Dict,
              finish_reason: Optional[str]) -> str:
    """Un evento `data: {...}`, línea en blanco incluida. Dado -- la forma es el contrato."""
    payload = {
        "id": chunk_id,
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "delta": delta, "finish_reason": finish_reason}],
    }
    return f"data: {json.dumps(payload)}\n\n"


def sse_events(req: ChatRequest):
    """
    Va entregando el stream de eventos de un request.

    Orden: un delta con el rol, después deltas de contenido, después un delta vacío que
    lleva finish_reason, y al final `data: [DONE]`.
    """
    chunk_id = f"chatcmpl-{uuid.uuid4().hex[:24]}"         # el mismo en todos los chunks
    yield sse_chunk(chunk_id, req.model, {"role": "assistant"}, None)

    produced, sent = [], ""
    for tok in stream_tokens(SERVE_SESS, render_prompt(req.messages), **decode_args(req)):
        produced.append(tok)
        text = tokenizer.decode(produced)
        if text.endswith("�"):
            continue          # medio carácter UTF-8: se manda cuando llegue el resto
        delta, sent = text[len(sent):], text
        if delta:
            yield sse_chunk(chunk_id, req.model, {"content": delta}, None)

    rest = tokenizer.decode(produced)[len(sent):]           # lo que haya quedado retenido
    if rest:
        yield sse_chunk(chunk_id, req.model, {"content": rest}, None)
    finish = "length" if len(produced) >= req.max_tokens else "stop"
    yield sse_chunk(chunk_id, req.model, {}, finish)
    yield "data: [DONE]\n\n"

## Levantalo

La ruta se registra acá, una sola vez, y despacha según `req.stream` a las dos cosas que
escribiste. Después `uvicorn` corre en un thread de fondo para que el notebook siga usable,
atado a `127.0.0.1`, así que nada fuera de esta máquina lo alcanza.

Los dos self-checks llaman al endpoint por HTTP de verdad y leen el contrato de vuelta.

In [16]:
@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    """La única ruta. La Consigna VI responde de una, la VII responde de a pedazos."""
    if req.stream:
        return StreamingResponse(sse_events(req), media_type="text/event-stream")
    return build_response(req)


PORT = 8000
_config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="error")
server = uvicorn.Server(_config)
threading.Thread(target=server.run, daemon=True).start()

for _ in range(200):                      # toma el socket en bastante menos de un segundo
    if server.started:
        break
    time.sleep(0.05)
print(f"sirviendo en http://127.0.0.1:{PORT}  (started={server.started})")


def check_api():
    url = f"http://127.0.0.1:{PORT}/v1/chat/completions"
    body = {"model": "tinygpt", "temperature": 0.0, "max_tokens": 16,
            "messages": [{"role": "user", "content": "who said: Now is the winter"}]}
    r = requests.post(url, json=body, timeout=60)
    assert r.status_code == 200, f"{r.status_code}: {r.text[:200]}"
    data = r.json()

    for key in ("id", "object", "created", "model", "choices", "usage"):
        assert key in data, f"a la respuesta le falta {key!r}"
    assert data["object"] == "chat.completion", data["object"]
    choice = data["choices"][0]
    assert choice["message"]["role"] == "assistant", choice["message"]
    assert choice["finish_reason"] in ("stop", "length"), choice["finish_reason"]

    usage = data["usage"]
    assert usage["prompt_tokens"] + usage["completion_tokens"] == usage["total_tokens"], usage
    assert usage["completion_tokens"] > 0, "no se generó nada"

    content = choice["message"]["content"]
    for tok in (USER, ASSISTANT, PAD):
        assert tok not in content, f"{tok} se filtró en la respuesta -- suprimí SCAFFOLD_IDS"

    # max_tokens es una promesa, no una sugerencia.
    short = requests.post(url, json={**body, "max_tokens": 2}, timeout=60).json()
    assert short["usage"]["completion_tokens"] <= 2, short["usage"]
    if short["choices"][0]["finish_reason"] == "length":
        assert short["usage"]["completion_tokens"] == 2, short["usage"]

    # La temperatura tiene que llegar de verdad al sampler. Greedy dos veces da lo mismo;
    # samplear caliente varias veces no puede dar siempre lo mismo.
    greedy = {requests.post(url, json={**body, "temperature": 0.0}, timeout=60)
              .json()["choices"][0]["message"]["content"] for _ in range(2)}
    assert len(greedy) == 1, "temperature=0 tiene que ser determinista"
    hot = {requests.post(url, json={**body, "temperature": 2.0}, timeout=60)
           .json()["choices"][0]["message"]["content"] for _ in range(6)}
    assert len(hot) > 1, "temperature=2.0 dio una sola respuesta -- ¿llega a sample_next?"

    # Un segundo turno tiene que llegar al modelo: una API de chat que se olvida la
    # conversación está rota, no es una decisión de diseño.
    multi = requests.post(url, json={**body, "messages": [
        {"role": "user", "content": "who said: Now is the winter"},
        {"role": "assistant", "content": "GLOUCESTER"},
        {"role": "user", "content": "continue: To be, or not"}]}, timeout=60).json()
    assert multi["usage"]["prompt_tokens"] > usage["prompt_tokens"], \
        "una conversación de varios turnos no produjo más tokens de prompt que un solo mensaje"

    # El rol system es una decisión de diseño, no una regla -- esto informa, no verifica.
    sys_usage = requests.post(url, json={**body, "messages": [
        {"role": "system", "content": "You are a Shakespearean actor."},
        {"role": "user", "content": "who said: Now is the winter"}]},
        timeout=60).json()["usage"]
    reached = sys_usage["prompt_tokens"] > usage["prompt_tokens"]

    print(f"contenido    {content!r}")
    print(f"finish       {choice['finish_reason']}")
    print(f"usage        {usage['prompt_tokens']}+{usage['completion_tokens']}"
          f"={usage['total_tokens']}")
    print(f"rol system   {'llega al modelo' if reached else 'se descarta'}"
          f"  ({usage['prompt_tokens']} -> {sys_usage['prompt_tokens']} tokens de prompt)"
          f"  -- defendé tu decisión en las preguntas")
    print("todos los checks pasaron")


def check_stream():
    url = f"http://127.0.0.1:{PORT}/v1/chat/completions"
    body = {"model": "tinygpt", "temperature": 0.0, "max_tokens": 16, "stream": True,
            "messages": [{"role": "user", "content": "continue: To be, or not to be,"}]}
    r = requests.post(url, json=body, stream=True, timeout=60)
    assert r.status_code == 200, f"{r.status_code}: {r.text[:200]}"
    assert "text/event-stream" in r.headers.get("content-type", ""), r.headers

    events = [l[len("data: "):] for l in r.iter_lines(decode_unicode=True)
              if l and l.startswith("data: ")]
    assert events, "no llegó ningún evento data:"
    assert events[-1] == "[DONE]", f"el stream termina con {events[-1]!r}, no con [DONE]"

    chunks = [json.loads(e) for e in events[:-1]]
    assert all(c["object"] == "chat.completion.chunk" for c in chunks), \
        "mientras se streamea, object tiene que ser chat.completion.chunk"
    assert len({c["id"] for c in chunks}) == 1, "todos los chunks necesitan el mismo id"
    assert chunks[0]["choices"][0]["delta"].get("role") == "assistant", \
        "el primer delta anuncia el rol"
    assert all(c["choices"][0]["finish_reason"] is None for c in chunks[:-1]), \
        "solo el último chunk lleva finish_reason"
    assert chunks[-1]["choices"][0]["finish_reason"] in ("stop", "length"), chunks[-1]

    text = "".join(c["choices"][0]["delta"].get("content", "") for c in chunks)
    assert text, "no se streameó ningún contenido"
    for tok in (USER, ASSISTANT, PAD):
        assert tok not in text, f"{tok} se filtró en el stream"
    assert "�" not in text, \
        "carácter de reemplazo -- estás decodificando los tokens de a uno"

    print(f"chunks       {len(chunks)} + [DONE]")
    print(f"streameado   {text!r}")
    print(f"finish       {chunks[-1]['choices'][0]['finish_reason']}")
    print("todos los checks pasaron")


check_api()
print()
check_stream()

sirviendo en http://127.0.0.1:8000  (started=True)


contenido    ',\nAnd'
finish       stop
usage        9+3=12
rol system   se descarta  (9 -> 9 tokens de prompt)  -- defendé tu decisión en las preguntas
todos los checks pasaron

chunks       5 + [DONE]
streameado   '\nAnd,'
finish       stop
todos los checks pasaron


# Consigna VIII — llamalo con el cliente de OpenAI

Los self-checks hablaron HTTP crudo. Esta es la parte que justifica todo el laburo: el
**cliente oficial `openai`**, la misma librería que le habla a GPT-5, apuntada a tu laptop.

No se modifica nada del cliente. Manda su request de siempre, parsea la respuesta a su
objeto `ChatCompletion` de siempre, y nunca se entera de que detrás de la URL hay un modelo
de 6.5M de parámetros corriendo en una CPU.

Hacelo de las dos formas:

- **completo** — unos cuantos mensajes, incluyendo uno que frene en `<|end|>` y otro que se
  quede sin `max_tokens`, leyendo los dos valores de `finish_reason`;
- **streaming** — con `stream=True`, imprimiendo cada `chunk.choices[0].delta.content` a
  medida que llega. El cliente te entrega objetos `ChatCompletionChunk` parseados del SSE
  que escribiste en la Consigna VII.

Si el texto streameado y el completo pueden diferir para el mismo prompt, explicá por qué
en tu informe.

In [17]:
from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="not-needed")


def show(r) -> None:
    c, u = r.choices[0], r.usage
    print(f"  {c.message.content!r}")
    print(f"  finish_reason={c.finish_reason} | usage {u.prompt_tokens}+{u.completion_tokens}"
          f"={u.total_tokens} | {type(r).__name__}")


# a) completo
print("a) completo\n")
calls = [
    ("who_said, greedy -- frena en <|end|>",
     dict(messages=[{"role": "user", "content": "who said: Now is the winter of our discontent"}],
          temperature=0, max_tokens=16)),
    ("continue, max_tokens=2 -- se queda sin tokens",
     dict(messages=[{"role": "user", "content": "continue: To be, or not to be,"}],
          temperature=0, max_tokens=2)),
    ("reverse, greedy",
     dict(messages=[{"role": "user", "content": "reverse: sword"}], temperature=0, max_tokens=8)),
    ("dos turnos, sampleado con temperature=0.8 y top_p=0.9",
     dict(messages=[{"role": "user", "content": "who said: Now is the winter"},
                    {"role": "assistant", "content": "GLOUCESTER"},
                    {"role": "user", "content": "continue: Now is the winter of our discontent"}],
          temperature=0.8, top_p=0.9, max_tokens=16)),
]
for title, kw in calls:
    print(f"> {title}")
    show(client.chat.completions.create(model="tinygpt", **kw))
    print()

# b) streaming
print("b) streaming\n")
stream_kw = dict(model="tinygpt", temperature=0, max_tokens=24,
                 messages=[{"role": "user", "content": "who said: Now is the winter of our discontent"}])
deltas, finish, types = [], None, set()
for chunk in client.chat.completions.create(stream=True, **stream_kw):
    types.add(type(chunk).__name__)
    piece = chunk.choices[0].delta.content
    if piece:
        deltas.append(piece)
        print(piece, end="", flush=True)
    finish = chunk.choices[0].finish_reason or finish
print(f"\n\n  {len(deltas)} deltas: {deltas}")
print(f"  finish_reason={finish} | objetos {types}")

full = client.chat.completions.create(stream=False, **stream_kw).choices[0].message.content
print(f"  greedy: streameado == completo -> {''.join(deltas) == full}")

# c) la alternativa descartada para el rol system (Consigna VI): pegarlo adelante del pedido.
print("\nc) system pegado adelante del pedido, en vez de descartarlo (greedy, sin el server)\n")
for body in ("who said: Now is the winter of our discontent",
             "You are a Shakespearean actor. who said: Now is the winter of our discontent"):
    out = generate_onnx(kv_sess, f"{USER}{body}{ASSISTANT}", do_sample=False,
                        max_new_tokens=16, suppress=SCAFFOLD_IDS)
    print(f"  {body[:45]!r:<48} -> {out!r}")

# d) por qué decode_args pide min_new_tokens=1: la distribución del primer token.
print("\nd) el primer token de un who_said corto\n")
prompt = f"{USER}who said: Now is the winter{ASSISTANT}"
logits = run_session(fp32_sess, {"input_ids": np.array([tokenizer.encode(prompt)])})[0][0, -1]
logits[SCAFFOLD_IDS] = -np.inf
probs = np.exp(logits - logits.max())
probs /= probs.sum()
top = np.argsort(-probs)[:4]
print("  " + ", ".join(f"{tokenizer.decode([int(i)])!r} {probs[i]:.3f}" for i in top))
print(f"  greedy sin mínimo: {generate_onnx(kv_sess, prompt, do_sample=False, suppress=SCAFFOLD_IDS)!r}")
print(f"  greedy con mínimo: {generate_onnx(kv_sess, prompt, do_sample=False, suppress=SCAFFOLD_IDS, min_new_tokens=1)!r}")

a) completo

> who_said, greedy -- frena en <|end|>
  'DUKE VINCENTIO'
  finish_reason=stop | usage 12+7=19 | ChatCompletion

> continue, max_tokens=2 -- se queda sin tokens
  '\nAnd'
  finish_reason=length | usage 12+2=14 | ChatCompletion

> reverse, greedy
  'gn'
  finish_reason=stop | usage 5+1=6 | ChatCompletion

> dos turnos, sampleado con temperature=0.8 y top_p=0.9
  'F'
  finish_reason=stop | usage 26+1=27 | ChatCompletion

b) streaming

D

U

KE

 V

INC

ENT

IO



  7 deltas: ['D', 'U', 'KE', ' V', 'INC', 'ENT', 'IO']
  finish_reason=stop | objetos {'ChatCompletionChunk'}
  greedy: streameado == completo -> True

c) system pegado adelante del pedido, en vez de descartarlo (greedy, sin el server)

  'who said: Now is the winter of our discontent'  -> 'DUKE VINCENTIO'
  'You are a Shakespearean actor. who said: Now '  -> ''

d) el primer token de un who_said corto

  '<|end|>' 0.222, ',' 0.041, '.' 0.020, 'B' 0.013
  greedy sin mínimo: ''
  greedy con mínimo: ',\nAnd'


## Informe — Consignas VI a VIII

**El rol `system` se descarta.** El self-check lo reporta: 9 tokens de prompt con y sin system. El modelo nunca vio un system prompt, y en el TP-2 la posición de la respuesta rutea mirando la palabra de la tarea justo después de `<|user|>`. La parte (c) de arriba muestra qué pasa si en vez de descartarlo se lo pega adelante del pedido: `who said: Now is the winter of our discontent` responde `'DUKE VINCENTIO'`, y con `You are a Shakespearean actor.` adelante responde `''`. Las alternativas están en la pregunta 3.

**El server no devuelve respuestas vacías.** `decode_args` pide `min_new_tokens=1`: `<|end|>` no puede ser el primer token, igual que `min_tokens` en vLLM o `min_new_tokens` en HuggingFace. Lo agregué porque `check_api` falló la primera vez con "no se generó nada".
- La parte (d) muestra por qué. Para el prompt corto del self-check, `who said: Now is the winter`, el token más probable es `<|end|>`, con 0,222; el segundo es `,`, con 0,041.
- Los `who_said` del TP-2 siempre tenían el parlamento completo (~25 tokens). Ante uno de 5 palabras, el modelo lo más probable es que termine sin contestar, y greedy devuelve `''`.
- Una respuesta vacía no le sirve a ninguna interfaz de chat, así que es una política razonable para la API. Pero hay que decirlo: lo que sale en su lugar (`',\nAnd'`) es basura. La política esconde que el modelo no sabe contestar ese pedido; no lo arregla.
- `stream_tokens` sigue teniendo `min_new_tokens=0` por defecto, así que la Consigna III no cambia.

**El texto streameado y el completo** salieron idénticos con `temperature=0`: los dos usan el mismo `stream_tokens` y el greedy es determinista. Con `temperature > 0` pueden diferir para el mismo prompt, y no por el streaming: cada request samplea con un generador nuevo sin semilla, porque la API no recibe `seed`. Dos requests no-stream con muestreo también dan textos distintos. Lo que el streaming sí podría cambiar es la decodificación del texto, si se mandaran pedazos de carácter. Por eso `sse_events` retiene el delta mientras termina en `�` y manda lo que quede al final: el texto concatenado es el mismo que devuelve `tokenizer.decode` sobre todos los tokens.

# Preguntas

1. `finish_reason` distingue `"stop"` de `"length"`. ¿Qué hace distinto quien llama en cada
   caso, y qué se rompe en una interfaz de chat si devolvés siempre `"stop"`?
2. `usage` reporta cuentas de tokens para un modelo que nadie está facturando. Nombrá dos
   cosas además de la facturación para las que quien llama usa esos números. ¿Por qué no
   está mientras se hace streaming, y qué le cuesta eso a quien llama?
3. Tu `render_prompt` tuvo que decidir qué significa un mensaje `system` para un modelo que
   nunca se fine-tuneó con uno. Decí qué elegiste y qué costarían las alternativas. ¿Qué
   haría falta para soportarlo en serio?
4. El streaming devuelve los mismos tokens en el mismo orden y no cambia ninguna métrica de
   la Consigna V — el tiempo total es idéntico. Nombrá qué sí mejora, en un término que un
   usuario reconocería, y explicá por qué importa más con 500 tokens que con 24.
5. El código de streaming tuvo que decodificar toda la respuesta en cada paso en vez de
   token por token. Explicá la falla usando BPE a nivel de bytes, y conectala con por qué
   la tarea `reverse` del TP-2 era imposible.
6. Este server atiende un request por vez, en Python, sin batching ni autenticación. Elegí
   **una** sola cosa que agregarías primero para un despliegue real y justificala contra
   tus números de la Consigna V.
7. En este notebook nunca cambiaste el modelo, solo cómo está representado y cómo se lo
   alcanza. Argumentá a favor o en contra: "la parte difícil del despliegue no es el
   modelo".

---

**Respuestas**

1. **`"stop"` quiere decir "terminé", y `"length"`, "me cortaron".**
   - Con `"stop"`, quien llama muestra la respuesta como completa y espera el próximo turno.
   - Con `"length"` sabe que la respuesta está truncada. Puede pedir la continuación (reenviar la conversación con la respuesta parcial como mensaje `assistant`), reintentar con más `max_tokens` o avisarle al usuario con un botón de "continuar". Si esperaba JSON, sabe que lo que recibió está incompleto y no lo parsea.

   Si el server devuelve siempre `"stop"`, la interfaz presenta como terminada una respuesta cortada a mitad de frase, sin aviso, y la lógica de continuación automática nunca se dispara. En la Consigna VIII, `max_tokens=2` devolvió `'\nAnd'` con `"length"`; reportado como `"stop"`, el cliente lo tomaría como la respuesta entera.

2. **Además de la facturación, quien llama usa `usage` para:**
   - **Manejar la ventana de contexto:** saber cuánto ocupa la conversación para recortar o resumir el historial antes de pasarse. Acá es crítico: el pedido de dos turnos de la Consigna VIII ya usa 26 de los 32 tokens de `block_size`.
   - **Aplicar límites y monitorear:** cuotas de tokens por minuto por usuario, latencia por token, capacidad.
   - **Detectar truncamiento:** `completion_tokens == max_tokens`.

   **En streaming no está porque no se conoce hasta el final,** y los chunks se mandan antes. El formato no tiene un mensaje de resumen por defecto (OpenAI lo agrega sólo si se pide con `stream_options`). A quien llama le cuesta contar los tokens por su cuenta, para lo cual necesita el mismo tokenizer y el mismo template que el server. Si no, se queda sin números exactos.

3. **Elegí descartarlo.** El modelo nunca vio un system prompt, y en el TP-2 el ruteo depende de la palabra de la tarea justo después de `<|user|>`. Las alternativas y lo que cuestan:
   - **Pegarlo adelante del primer mensaje del usuario.** Rompe el ruteo: la parte (c) de la Consigna VIII pasa de `'DUKE VINCENTIO'` a `''`. Además gasta parte de una ventana de 32 tokens: `You are a Shakespearean actor.` son 7.
   - **Un turno `<|user|>` aparte.** El modelo lo trataría como un pedido y respondería algo; además gasta ventana.
   - **Rechazar el request con un 400.** Es honesto, pero rompe a todos los clientes que mandan un system prompt por defecto, que son la mayoría de los SDKs y herramientas.

   Descartarlo tiene su propio costo: quien llama cree que su instrucción se aplicó, y por eso hay que documentarlo. Soportarlo en serio pide dos cosas. Un token `<|system|>` en el template y un fine-tune con ejemplos donde el system prompt cambie el comportamiento. Y una ventana bastante más grande que 32 tokens, donde entren el system prompt, la pregunta y la respuesta.

4. **Mejora el tiempo hasta el primer token: la respuesta empieza a aparecer enseguida en vez de después de una pausa en blanco.** El tiempo total no cambia, pero la latencia percibida pasa a ser la del primer token, que no depende del largo. Con 24 tokens a ~0,6 ms/token la espera total es de 15 ms, y streamear no se nota. Con 500 tokens en un modelo real a ~30 tokens/s son ~17 segundos frente a una pantalla vacía, contra las primeras palabras en una fracción de segundo. Mientras tanto el usuario lee a la par y puede cortar si la respuesta no va por donde quería.

5. **El BPE de GPT-2 trabaja sobre bytes, no sobre caracteres.** Un carácter que no es ASCII (`é`, `—`, un emoji, o la `а` cirílica que apareció en la muestra con temperatura 2,0) ocupa de 2 a 4 bytes en UTF-8. El BPE puede dejarlo partido entre dos tokens. `tokenizer.decode([un_id])` sobre un token que tiene sólo el primer byte no puede formar un carácter válido y devuelve `�`. Por eso hay que decodificar toda la secuencia y mandar sólo el sufijo nuevo, reteniéndolo mientras termine en `�`.

   **Es la misma falla que hacía imposible `reverse`:** la unidad con la que trabaja el tokenizer no está alineada con la que necesita la tarea. Invertir pide los caracteres que quedan *adentro* de un token; streamear pide caracteres completos, que pueden quedar *partidos entre* dos tokens. Un token no es ni un carácter ni una palabra.

6. **Agregaría batching.** La justificación está en la Consigna V.
   - Con cache, el paso es un producto matriz-vector limitado por la lectura de los 12,9 MB de la cabeza.
   - El grafo sin cache calcula 32 filas por llamada y tarda sólo 1,9 veces lo que el de cache tarda con 1 (1,186 contra 0,613 ms).
   - Agrupar las filas de varios requests en una sola llamada lee los pesos una vez y amortiza los nodos de ONNX Runtime, el `run()` y la parte fija de Python (~0,11 ms/token). Con 32 requests concurrentes, el throughput subiría cerca de un orden de magnitud con el mismo hardware.
   - Hoy el server atiende de a un request y a ~1.600 tokens/s; esa es toda su capacidad, la use un usuario o cien.

   Hacerlo bien exige padding o batching continuo, y un cache por request. La autenticación es obligatoria antes de exponerlo fuera de `127.0.0.1`, pero no cambia ninguno de los números de la Consigna V.

7. **Mayormente a favor, con un límite.**
   - **A favor.** En este notebook el modelo no cambió, y todo lo que lo volvió desplegable fue representación e interfaz: el export, el cache como entrada y salida, la cuantización, el template, el contrato de la API, `finish_reason`, `usage`, el streaming y la supresión del andamiaje. Los mismos pesos pasaron de 2,19 a 0,42 ms/token, 5,2 veces más rápido, y quedaron 4 veces más chicos. Todos los bugs con los que me crucé fueron de despliegue: el export que deja el modelo en `train`, la máscara que se puede hornear como constante, el UTF-8 partido, el rol `system`, la respuesta vacía. Ninguno se arregla entrenando mejor.
   - **El límite.** El modelo pone el techo, y el despliegue no lo puede subir. Ninguna de estas técnicas hace que `who_said` funcione, porque en el TP-2 estaba cerca del azar. La política de "nunca una respuesta vacía" esconde que el modelo no sabe contestar, no lo arregla.

   La parte difícil del despliegue no es el modelo, pero lo que el usuario recibe sigue siendo el modelo.

## Bajalo

El thread es daemon, así que muere con el kernel — pero dejar un socket tomado mientras
seguís trabajando es la forma más rápida de que la próxima corrida te salude con *address
already in use*.

In [18]:
server.should_exit = True
time.sleep(1.0)
print("server detenido")

try:                                  # el visor de Netron, si levantaste uno
    import netron
    netron.stop()
except Exception:
    pass

print("\nartefactos en ./onnx:")
for f in sorted(os.listdir(ONNX_DIR)):
    print(f"  {f:<24} {os.path.getsize(os.path.join(ONNX_DIR, f)) / 1e6:>7.1f} MB")

server detenido

artefactos en ./onnx:
  tinygpt.int8.kv.onnx         6.6 MB
  tinygpt.int8.onnx            6.6 MB
  tinygpt.kv.onnx             26.2 MB
  tinygpt.onnx                26.2 MB


# Conclusiones

- **Un grafo trazado que corre no es un grafo correcto; la paridad en varios largos es la única prueba.**
  - El export con y sin KV-cache reproduce a PyTorch con un error de ~1e-5 en todos los largos de secuencia y de pasado.
  - Las dos trampas existen: el modo entrenamiento que `export()` deja puesto y la máscara que se puede hornear como constante.
  - El grafo de 221 nodos muestra que más de la mitad no es aritmética del modelo: son cuentas de formas que el tracer convirtió en operaciones.
- **En este modelo todo lo decide el vocabulario.** El embedding y la cabeza son el 98 % de los pesos.
  - Por eso INT8 lo hace 4 veces más chico sin cambiar el comportamiento: +0,0018 nats, 100 % de coincidencia en el siguiente token, aunque la paridad cruda dé 0,38.
  - Y por eso es apenas 1,3 veces más rápido.
  - El KV-cache da 1,9 veces y no 32, porque cada paso igual lee la cabeza entera y paga un costo fijo de engine y de Python.
  - Combinados dan 5,2 veces PyTorch.
- **La API es el producto.** El mismo cliente oficial de OpenAI le habla al modelo sin saber qué hay detrás, en modo completo y en streaming. Los detalles que lo hacen funcionar no tienen nada que ver con el modelo:
  - `finish_reason`, `usage` y la supresión del andamiaje;
  - retener los pedazos de carácter del BPE;
  - qué hacer con un `system` que el modelo nunca vio;
  - qué hacer cuando lo más probable es una respuesta vacía.

# ¡Felicitaciones! 🎉

A lo largo de tres notebooks preentrenaste un GPT, lo hiciste sparse con una Mixture of
Experts, le enseñaste a seguir instrucciones con fine-tuning completo y con LoRA, lo
exportaste a un grafo portable, lo cuantizaste, y lo serviste detrás de la misma API que
habla el resto de la industria.

Preentrenamiento, alineación y despliegue — el pipeline entero, a una escala que podés leer
de punta a punta.